# All the results side by side

Every run saves its own files on Drive (`cdt-data/results_v2/<run>/`): the scores, every epoch
of training and the test predictions. So even when Colab disconnects or a new session starts,
nothing is lost. This notebook just reads them back and puts them together.

No GPU and no data copy needed, it takes about a minute (Runtime type can stay on CPU).
Run it again after each new seed and it updates everything.

Everything here is aggregate (scores, curves, confusion matrices), no participant IDs and no
drawings, so the table and the figures can go in the report and on slides.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/cdt-alzheimer-screening'
if not os.path.exists(REPO):
    !git clone -q https://github.com/wiambenadder/cdt-alzheimer-screening.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1

RESULTS = '/content/drive/MyDrive/cdt-data/results_v2'
os.environ['CDT_RESULTS_DIR'] = RESULTS

In [ ]:
from src.report import make_report
from IPython.display import Image, display
import pandas as pd
pd.set_option('display.width', 200)

r = make_report(RESULTS)   # writes everything to cdt-data/results_v2/report/

print('Every run:')
display(r['runs'][['run', 'n_test', 'best_epoch', 'kappa', 'kappa_lo', 'kappa_hi',
                   'tuned_kappa', 'tuned_lo', 'tuned_hi', 'accuracy', 'macro_f1']].round(3))
if not r['seeds'].empty:
    print('Mean over the seeds:')
    display(r['seeds'].round(3))
if not r['comparisons'].empty:
    print('Paired comparisons (if the interval does not cross 0, the difference is real):')
    display(r['comparisons'].round(3))

## The figures

* **kappa_ci**: every run with its 95% interval, plain and with the tuned cut points
* **training_curves**: validation kappa and loss for every epoch of every seed, the star is the epoch that was kept
* **confusion_matrices**: for each true score, how often each score was predicted

They are saved as PNG in `cdt-data/results_v2/report/`, with the tables as CSV.

In [ ]:
for f in r['figures']:
    print(f.name)
    display(Image(str(f), width=900))